# 🧪 2. Preprocess Input & Dataset Consolidation

Este notebook representa la segunda fase del pipeline de datos de **ProtFlash**. Su objetivo es consolidar los archivos CSV individuales en conjuntos de datos estructurados y etiquetados para el aprendizaje supervisado.

---

## 📋 Descripción General
El script automatiza la carga de múltiples archivos CSV (generados a partir de archivos FASTA), asigna etiquetas de clase basadas en el nombre del archivo y genera los archivos maestros de **entrenamiento, validación y prueba**.


## 🛠️ Funcionalidades Principales

1.  **Gestión de Rutas:**
    * Configura el acceso a los directorios de datos procesados (`/csv`) y prepara las rutas para los futuros embeddings.
2.  **Etiquetado Automático:**
    * Implementa una lógica de detección de clase:
        * Asigna `clase = 0` si el nombre del archivo contiene "neg" (secuencias negativas).
        * Asigna `clase = 1` si el nombre del archivo contiene "pos" (secuencias positivas).
3.  **Consolidación de Splits:**
    * Combina los datos dispersos para formar tres datasets principales:
        * **Train:** Combinación de `train_neg` y `train_pos`.
        * **Val:** Combinación de `val_neg` y `val_pos`.
        * **Test:** Combinación de `test_neg` y `test_pos`.
4.  **Generación de Archivos Maestros:**
    * Exporta los resultados a nuevos archivos CSV (`train.csv`, `val.csv`, `test.csv`) con una estructura estandarizada de tres columnas: `id`, `sequence` y `label`.

## 📊 Resumen del Dataset
El proceso consolida un total de **39,096 secuencias**:
* **Train:** 19,548 secuencias.
* **Val:** 5,125 secuencias.
* **Test:** 15,685 secuencias.

Configuración de rutas de entrada y salida

In [ ]:
INPUT_CSV = "/content/drive/MyDrive/Experimentos/ProtFlash/Data/csv"
OUTPUT_EMBEDDINGS = "/content/drive/MyDrive/Experimentos/ProtFlash/Embeddings"

Carga y procesamiento de archivos CSV

In [ ]:
import os
import pandas as pd

print("📂 Cargando CSVs desde:", INPUT_CSV)
print("=" * 50)

📂 Cargando CSVs desde: /content/drive/MyDrive/ProtFlash-2102/Data/csv


In [ ]:
datasets = {}

for csv_file in sorted(os.listdir(INPUT_CSV)):
    if csv_file.endswith('.csv'):
        csv_path = os.path.join(INPUT_CSV, csv_file)
        df = pd.read_csv(csv_path)

        # Detectar columnas automáticamente
        # Buscar columna de ID
        seq_col = 'sequence'

        # Crear lista de tuplas (id, sequence)
        nombre = os.path.splitext(csv_file)[0]
        clase = 0
        if nombre.find("neg") > -1:
          clase = 0
        else:
          clase = 1

        data = [(f"{nombre}_{i}", str(row[seq_col]),clase) for i, (_, row) in enumerate(df.iterrows())]

        datasets[nombre] = data

        print(f"✅ {csv_file}: {len(data)} secuencias")
        print(f"   Columnas: {list(df.columns)}")
        print(f"   Ejemplo: (\"{data[0][0]}\", \"{data[0][1][:50]}...\"  \"{data[0][2]}\" ...)")
        print()

✅ test_neg.csv: 10771 secuencias
   Columnas: ['id', 'description', 'sequence', 'length']
   Ejemplo: ("test_neg_0", "RTASGLPFMKMHGLGNDFVVIDAREGDDPVTPA..."  "0" ...)

✅ test_pos.csv: 4914 secuencias
   Columnas: ['id', 'description', 'sequence', 'length']
   Ejemplo: ("test_pos_0", "KRWWKWWRR..."  "1" ...)

✅ train_neg.csv: 9767 secuencias
   Columnas: ['id', 'description', 'sequence', 'length']
   Ejemplo: ("train_neg_0", "NMLFIKNANNVKSLIKGECTLSVEIDGKEFIINGIKIAKRPEDRVGMKIQ..."  "0" ...)

✅ train_pos.csv: 9781 secuencias
   Columnas: ['id', 'description', 'sequence', 'length']
   Ejemplo: ("train_pos_0", "GLFDIIKNIFSGL..."  "1" ...)

✅ val_neg.csv: 2561 secuencias
   Columnas: ['id', 'description', 'sequence', 'length']
   Ejemplo: ("val_neg_0", "DYNPTGDKDAPVYACLV..."  "0" ...)

✅ val_pos.csv: 2564 secuencias
   Columnas: ['id', 'description', 'sequence', 'length']
   Ejemplo: ("val_pos_0", "YSLQMGATAIKQVKKLFKKWGW..."  "1" ...)



In [ ]:
datasets.keys()

dict_keys(['test_neg', 'test_pos', 'train_neg', 'train_pos', 'val_neg', 'val_pos'])

Combinación de splits (neg + pos)

In [ ]:
datasets['train'] = datasets['train_neg'] + datasets['train_pos']
datasets['val'] = datasets['val_neg'] + datasets['val_pos']
datasets['test'] = datasets['test_neg'] + datasets['test_pos']

Guardado de datasets consolidados

In [ ]:
import pandas

df = pd.DataFrame(datasets['train'], columns=['id', 'sequence','label'])
df.to_csv(INPUT_CSV+"/train.csv")

df = pd.DataFrame(datasets['val'], columns=['id', 'sequence','label'])
df.to_csv(INPUT_CSV+"/val.csv")

df = pd.DataFrame(datasets['test'], columns=['id', 'sequence','label'])
df.to_csv(INPUT_CSV+"/test.csv")